# Ch7. Time Series Regression
**Forecasting: Principles & Practice (Python Edition)**  
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

In [ ]:
%pip install statsforecast neuralforecast hierarchicalforecast mlforecast utilsforecast

## [Slide 5] Spurious Regression

In [ ]:
import statsmodels.api as sm

# Check residual autocorrelation
residuals = fitted_model.resid
sm.graphics.tsa.plot_acf(residuals, lags=20)

# Formal test
from statsmodels.stats.stattools import durbin_watson
dw = durbin_watson(residuals)
print(f"Durbin-Watson: {dw:.4f}")  # ~2 indicates no autocorrelation

## [Slide 8] Fourier Terms for Seasonality

In [ ]:
from utilsforecast.feature_engineering import fourier, trend

# Add K=3 Fourier pairs for m=52 (weekly data)
df_feat = fourier(df, freq=52, k=3)
df_feat = trend(df_feat)

## [Slide 12] Stepwise vs. Best Subset Selection

In [ ]:
from mlforecast import MLForecast
from sklearn.linear_model import LinearRegression
from utilsforecast.feature_engineering import fourier, trend

# Build feature matrix with trend + K Fourier pairs
df_feat = fourier(df, freq=4, k=2)
df_feat = trend(df_feat)

mf = MLForecast(
    models={"OLS": LinearRegression()},
    freq="QS-OCT",
    target_transforms=None,
)
mf.fit(df_feat, fitted=True, static_features=[])

## [Slide 15] Prediction Intervals for Regression

In [ ]:
# Prediction intervals using PredictionIntervals
from mlforecast.utils import PredictionIntervals

mf.fit(df_feat, fitted=True)
fc = mf.predict(
    h=4,
    X_df=future_data,
    level=[80, 95],
    prediction_intervals=PredictionIntervals(n_windows=10, h=4),
)

## [Slide 18] Piecewise Linear Trends

In [ ]:
import numpy as np

# Create piecewise linear features
tau1 = 2008   # GFC breakpoint
df["t"] = np.arange(1, len(df)+1)
df["t_break1"] = np.maximum(0, df["t"] - tau1)

## [Slide 21] Summary: Regression Modelling Workflow

In [ ]:
import statsmodels.formula.api as smf

# Full model with trend + quarterly dummies
mod = smf.ols("y ~ t + Q('Q1') + Q('Q2') + Q('Q3')", data=df).fit()
print(mod.summary())
sm.graphics.tsa.plot_acf(mod.resid, lags=20)